# Model V3 - RandomForest / SVM / XGBoost / CatBoost 비교

이 노트북은 `model_v1_1`의 전처리, split, feature 정책을 그대로 유지하고 모델 알고리즘만 바꿔 비교합니다.

비교 모델:

- `random_forest`: v1_1 기준 tree ensemble
- `linear_svm`: 대용량 데이터에서 실행 가능한 선형 SVM
- `xgboost`: 외부 패키지 설치 후 사용하는 gradient boosting tree
- `catboost`: 외부 패키지 설치 후 사용하는 categorical boosting tree

중요한 조건:

- 통합모델 feature에는 `JUDGE_SEX`를 넣지 않습니다.
- `gender_split`은 `암`, `거세`만 전용 모델을 만들고 `수`는 unified 예측을 fallback으로 사용합니다.
- SVM은 full kernel SVM이 아니라 `LinearSVC`입니다. 전체 행 수가 매우 커서 RBF/kernel SVM은 현실적으로 너무 느립니다.


## 0. 실행 설정

처음 실행할 때는 아래 설정을 확인하세요.

- `INSTALL_MISSING_PACKAGES=True`: `xgboost`, `catboost`가 없으면 노트북 안에서 설치합니다.
- `USE_SAMPLE=False`: 전체 데이터 실행입니다. 빠른 테스트만 할 때 `True`로 바꿉니다.
- `LINEAR_SVM_TRAIN_SAMPLE_N=500_000`: SVM은 너무 느릴 수 있어 기본적으로 train 일부만 사용합니다. 완전한 비교가 필요하면 `None`으로 바꿉니다.


In [1]:
from pathlib import Path
import json
import warnings
import sys
import subprocess
import importlib.util
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.svm import LinearSVC
from sklearn.base import BaseEstimator, ClassifierMixin, clone

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 160)
pd.set_option('display.max_rows', 120)

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

BASE_DIR = Path.cwd()
RANDOM_STATE = 42

USE_SAMPLE = False
SAMPLE_N = 200_000

ADD_AREA_FEATURES = True
ADD_LINEAGE_FEATURES = True
ADD_WEATHER_FEATURES = True
APPLY_SPATIAL_WEATHER_IMPUTE = True

WEATHER_WINDOWS = [30, 90, 180]
SPLIT_SEXES = ['암', '거세']
FALLBACK_SEXES = ['수']

# 외부 모델 설치/비교 설정
INSTALL_MISSING_PACKAGES = True
RUN_UNIFIED = True
RUN_GENDER_SPLIT = True
MODEL_CANDIDATES = [
    'random_forest',
    'linear_svm',
    'xgboost',
    'catboost',
]

RF_N_ESTIMATORS = 160
RF_MAX_DEPTH = 22
RF_MIN_SAMPLES_LEAF = 2
XGB_N_ESTIMATORS = 250
CATBOOST_ITERATIONS = 250

# SVM은 전체 데이터에서 시간이 오래 걸릴 수 있어 기본적으로 학습행을 제한합니다.
# 완전 동일 조건으로 비교하고 싶으면 None으로 바꾸세요.
LINEAR_SVM_TRAIN_SAMPLE_N = 500_000

TARGET = 'LAST_GRADE'
OUTPUT_DIR = BASE_DIR / 'model_v3_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

print('BASE_DIR:', BASE_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)


BASE_DIR: c:\Users\010hy\Desktop\2.university\02_데이터마이닝\project
OUTPUT_DIR: c:\Users\010hy\Desktop\2.university\02_데이터마이닝\project\model_v3_outputs


## 0-1. XGBoost / CatBoost 설치와 import

이 셀은 현재 파이썬 환경에 `xgboost`, `catboost`가 없으면 `pip install`을 실행합니다.

설치가 실패하면 노트북 전체를 멈추지는 않고, 해당 모델만 결과표에 실패로 기록합니다.


In [2]:
PACKAGE_SPECS = {
    'xgboost': 'xgboost',
    'catboost': 'catboost',
}


def ensure_package(module_name, package_name):
    if importlib.util.find_spec(module_name) is not None:
        return True
    if not INSTALL_MISSING_PACKAGES:
        print(f'[skip install] {module_name} is missing.')
        return False
    print(f'[install] {package_name}')
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])
        return importlib.util.find_spec(module_name) is not None
    except Exception as e:
        print(f'[install failed] {package_name}: {repr(e)}')
        return False


XGBClassifier = None
CatBoostClassifier = None

if ensure_package('xgboost', PACKAGE_SPECS['xgboost']):
    try:
        from xgboost import XGBClassifier
        import xgboost
        print('xgboost:', xgboost.__version__)
    except Exception as e:
        print('[import failed] xgboost:', repr(e))
        XGBClassifier = None

if ensure_package('catboost', PACKAGE_SPECS['catboost']):
    try:
        from catboost import CatBoostClassifier
        import catboost
        print('catboost:', catboost.__version__)
    except Exception as e:
        print('[import failed] catboost:', repr(e))
        CatBoostClassifier = None

print('XGBClassifier available:', XGBClassifier is not None)
print('CatBoostClassifier available:', CatBoostClassifier is not None)


xgboost: 3.0.2
[install] catboost
catboost: 1.2.10
XGBClassifier available: True
CatBoostClassifier available: True


## 1. 데이터 로드

현재 프로젝트 폴더 기준 상대 경로로 읽습니다.

- `hanwoo_train/hanwoo_train.csv`: 학습 데이터와 정답 `LAST_GRADE`
- `hanwoo_area.csv`: 농장별 면적과 연도별 사육두수
- `hanwoo_lineage/hanwoo_lineage.csv`: 혈통 중 KPN 정보
- `hanwoo_weather.csv`: 지점별 일별 기상 관측값
- `stn_station위치.csv`: 기상 지점 좌표와 고도

읽기 오류를 줄이기 위해 `utf-8-sig`, `utf-8`, `cp949` 순서로 시도합니다.

In [3]:
def read_csv_safe(path, **kwargs):
    path = Path(path)
    last_error = None
    for enc in ['utf-8-sig', 'utf-8', 'cp949']:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error

TRAIN_PATH = BASE_DIR / 'hanwoo_train' / 'hanwoo_train.csv'
AREA_PATH = BASE_DIR / 'hanwoo_area.csv'
WEATHER_PATH = BASE_DIR / 'hanwoo_weather.csv'
LINEAGE_PATH = BASE_DIR / 'hanwoo_lineage' / 'hanwoo_lineage.csv'
STATION_PATH = next(BASE_DIR.glob('stn_station*.csv'), None)

train = read_csv_safe(TRAIN_PATH, nrows=SAMPLE_N if USE_SAMPLE else None)
print('train:', train.shape)
display(train.head(2))

area = read_csv_safe(AREA_PATH) if ADD_AREA_FEATURES else None
print('area:', None if area is None else area.shape)

lineage = read_csv_safe(LINEAGE_PATH, usecols=['CATTLE_NO', 'KPN_NO']) if ADD_LINEAGE_FEATURES else None
print('lineage:', None if lineage is None else lineage.shape)

weather = read_csv_safe(WEATHER_PATH) if ADD_WEATHER_FEATURES else None
print('weather:', None if weather is None else weather.shape)

station = read_csv_safe(STATION_PATH) if (ADD_WEATHER_FEATURES and STATION_PATH is not None) else None
print('station:', None if station is None else station.shape)

train: (2408699, 23)


,sido,sigungu,eupmyeondong,stn,ABATT_DATE,JUDGE_DATE,JUDGE_SEX,WEIGHT,BACKFAT,REA,WINDEX,WGRADE,INSFAT,YUKSAK,FATSAK,TISSUE,GROWTH,COST_AMT,AGE,BIRTH_YMD,CATTLE_NO,FARM_UNIQUE_NO,LAST_GRADE
0,강원특별자치도,정선군,북평면,563,2023-01-01,2023-01-02,암,218,3.0,44.0,63.23,A,1.0,6.0,7.0,5.0,9.0,-99.0,76,20160915,mI4i8G0BJ8kWD6gdm77RmTzyuIx6N2ZMaZ7wFXx3xb4=,hqh4Qmh5g+ymbYtqjAeKeA==,3B
1,강원특별자치도,평창군,진부면,560,2023-01-01,2023-01-02,거세,520,12.0,100.0,61.46,B,4.0,5.0,3.0,3.0,3.0,-99.0,32,20200504,tMgfi1p35taO9GH4XN4x0bfO8czy79B8V9NyczRuV+8=,N5qW6dZ91QWCMHg/1kPdlA==,1B


area: (91896, 5)
lineage: (1809455, 2)
weather: (973248, 7)
station: (444, 12)


## 2. 기본 정리와 날짜 feature

공모전 자료에서 결측은 `-99`로 표시되어 있습니다. 문자열 `'-99'`, 숫자 `-99`, `-99.0`을 모두 결측으로 바꿉니다.

날짜에서는 도축연도, 도축월, 도축연월, 계절, 판정 지연일을 만듭니다. 이 feature들은 누수 위험이 낮고, 성별/월별 가격 보간과 농장 사육두수 선택에도 필요합니다.

In [4]:
MISSING_TOKENS = [-99, -99.0, '-99', '-99.0', '', ' ']

NUMERIC_BASE_COLS = [
    'WEIGHT', 'BACKFAT', 'REA', 'WINDEX', 'INSFAT', 'YUKSAK',
    'FATSAK', 'TISSUE', 'GROWTH', 'COST_AMT', 'AGE'
]
CARCASS_NUMERIC_COLS = ['BACKFAT', 'REA', 'WINDEX', 'INSFAT', 'YUKSAK', 'FATSAK', 'TISSUE', 'GROWTH']


def normalize_missing(df):
    return df.copy().replace(MISSING_TOKENS, np.nan)


def add_date_features(df):
    out = df.copy()
    out['ABATT_DATE'] = pd.to_datetime(out['ABATT_DATE'], errors='coerce')
    out['JUDGE_DATE'] = pd.to_datetime(out['JUDGE_DATE'], errors='coerce')
    out['BIRTH_YMD'] = pd.to_datetime(out['BIRTH_YMD'].astype('string'), format='%Y%m%d', errors='coerce')
    out['abatt_year'] = out['ABATT_DATE'].dt.year.astype('Int64')
    out['abatt_month'] = out['ABATT_DATE'].dt.month.astype('Int64')
    out['abatt_ym'] = out['ABATT_DATE'].dt.to_period('M').astype('string')
    out['judge_delay_days'] = (out['JUDGE_DATE'] - out['ABATT_DATE']).dt.days
    month = out['abatt_month'].astype('float')
    conditions = [month.isin([3, 4, 5]), month.isin([6, 7, 8]), month.isin([9, 10, 11])]
    choices = ['spring', 'summer', 'fall']
    out['abatt_season'] = np.select(conditions, choices, default='winter')
    return out


train = normalize_missing(train)
for col in NUMERIC_BASE_COLS + ['stn']:
    if col in train.columns:
        train[col] = pd.to_numeric(train[col], errors='coerce')
train = add_date_features(train)

display(train[[TARGET, 'JUDGE_SEX', 'ABATT_DATE', 'abatt_year', 'abatt_month', 'abatt_ym']].head())
print(train[TARGET].value_counts(dropna=False))

,LAST_GRADE,JUDGE_SEX,ABATT_DATE,abatt_year,abatt_month,abatt_ym
0,3B,암,2023-01-01,2023,1,2023-01
1,1B,거세,2023-01-01,2023,1,2023-01
2,1++B,거세,2023-01-01,2023,1,2023-01
3,1++B,거세,2023-01-01,2023,1,2023-01
4,1B,거세,2023-01-01,2023,1,2023-01


LAST_GRADE
1++B    319588
1+B     311222
1B      299290
1++A    207026
2B      195866
1A      168174
1+A     167246
1+C     130094
1++C    128997
2A      127410
1C      116976
3B       97576
2C       62938
3A       47214
3C       23542
등외        5540
Name: count, dtype: int64


## 3. Train / Validation / Test 분할

연도별 분할은 쓰지 않습니다. 전체 데이터를 `70% / 15% / 15%`로 나누고, `LAST_GRADE` 기준 stratified split을 적용합니다.

중앙값, 최빈값, KPN 빈도 등 전처리 기준값은 train split에서만 계산하고 validation/test에 적용합니다.

In [5]:
model_source = train.dropna(subset=[TARGET]).copy()

train_idx, temp_idx = train_test_split(
    model_source.index,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=model_source[TARGET]
)
valid_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=model_source.loc[temp_idx, TARGET]
)

parts = {
    'train': model_source.loc[train_idx].copy(),
    'valid': model_source.loc[valid_idx].copy(),
    'test': model_source.loc[test_idx].copy(),
}

for name, df in parts.items():
    print(name, df.shape)
    print(df[TARGET].value_counts(normalize=True).sort_index().round(4).to_string())
    print()

train (1686089, 28)
LAST_GRADE
1++A    0.0859
1++B    0.1327
1++C    0.0536
1+A     0.0694
1+B     0.1292
1+C     0.0540
1A      0.0698
1B      0.1243
1C      0.0486
2A      0.0529
2B      0.0813
2C      0.0261
3A      0.0196
3B      0.0405
3C      0.0098
등외      0.0023

valid (361305, 28)
LAST_GRADE
1++A    0.0859
1++B    0.1327
1++C    0.0536
1+A     0.0694
1+B     0.1292
1+C     0.0540
1A      0.0698
1B      0.1243
1C      0.0486
2A      0.0529
2B      0.0813
2C      0.0261
3A      0.0196
3B      0.0405
3C      0.0098
등외      0.0023

test (361305, 28)
LAST_GRADE
1++A    0.0859
1++B    0.1327
1++C    0.0536
1+A     0.0694
1+B     0.1292
1+C     0.0540
1A      0.0698
1B      0.1243
1C      0.0486
2A      0.0529
2B      0.0813
2C      0.0261
3A      0.0196
3B      0.0405
3C      0.0098
등외      0.0023



## 4. 농장 규모 feature

농장 면적/사육두수 데이터는 농장번호가 중복될 수 있으므로 먼저 `FARM_UNIQUE_NO` 단위 median으로 집계합니다.

- `cow_year`: 도축연도에 맞는 사육두수 선택
- `log_cow_year`: 사육두수의 오른쪽 긴 꼬리를 줄이기 위한 `log1p`
- `log_area`: 면적의 오른쪽 긴 꼬리를 줄이기 위한 `log1p`

In [6]:
def prepare_area_table(area_df):
    if area_df is None:
        return None
    out = normalize_missing(area_df)
    for col in ['C2023', 'C2024', 'C2025', 'AREA']:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    out.loc[out['AREA'] <= 0, 'AREA'] = np.nan
    return out.groupby('FARM_UNIQUE_NO', as_index=False)[['C2023', 'C2024', 'C2025', 'AREA']].median()


def merge_area_features(df, area_table):
    if area_table is None:
        return df.copy()
    out = df.merge(area_table, on='FARM_UNIQUE_NO', how='left')
    year = out['abatt_year'].astype('float')
    out['cow_year'] = np.select(
        [year.eq(2023), year.eq(2024), year.eq(2025)],
        [out['C2023'], out['C2024'], out['C2025']],
        default=np.nan
    )
    return out.drop(columns=['C2023', 'C2024', 'C2025'], errors='ignore')


area_table = prepare_area_table(area)
if area_table is not None:
    print('area_table:', area_table.shape)
    display(area_table.head())

for key in parts:
    parts[key] = merge_area_features(parts[key], area_table)

area_table: (86703, 5)


,FARM_UNIQUE_NO,C2023,C2024,C2025,AREA
0,++3dDT9h9GLLBVoQDgn1IA==,69.0,55.0,37.0,1056.00
1,++9y4LAgYZMHjJNqlquqOA==,9.0,8.0,6.0,90.00
2,++BS1viAh3hOdGNLUqO9AQ==,8.0,NaN,NaN,NaN
3,++EgHgT0FBhXP2UY4tsA2Q==,126.0,132.0,134.0,1040.00
4,++FegxXsnx/IotYuAWBj8g==,6.0,9.0,6.0,368.34


## 5. 혈통 KPN 요약 feature

혈통 데이터는 모든 개체에 매칭되지 않습니다. 그래서 1차 모델에서는 복잡한 혈통 구조 대신 안전한 KPN 요약만 씁니다.

- `kpn_known`: 혈통 데이터에 매칭되고 KPN 값이 있으면 1
- `kpn_freq`: train split에서 해당 KPN이 등장한 빈도
- `log_kpn_freq`: `log1p(kpn_freq)`

In [7]:
def prepare_lineage_table(lineage_df):
    if lineage_df is None:
        return None
    out = normalize_missing(lineage_df[['CATTLE_NO', 'KPN_NO']].copy())
    return out.drop_duplicates('CATTLE_NO', keep='first')


def merge_lineage_features(df, lineage_table):
    if lineage_table is None:
        out = df.copy()
        out['KPN_NO'] = np.nan
        return out
    return df.merge(lineage_table, on='CATTLE_NO', how='left')


lineage_table = prepare_lineage_table(lineage)
if lineage_table is not None:
    print('lineage_table:', lineage_table.shape)
    display(lineage_table.head())

for key in parts:
    parts[key] = merge_lineage_features(parts[key], lineage_table)

lineage_table: (1809455, 2)


,CATTLE_NO,KPN_NO
0,yaSF9EHVPBIFeOdxL/st19so274mMHAbxmME5JhQ0Jk=,mgzxMpKRCOrwfHqsCHzP47qdLKwCfR8ULPzyRiJVOkM=
1,iCeTx0O5z/M8GAcglblMDFVCzmvnkdPVRKTQtIN4ZFg=,mgzxMpKRCOrwfHqsCHzP47qdLKwCfR8ULPzyRiJVOkM=
2,vI7joD/5v0T+wqvjC0d/36Sv5y+9NG18W+y2/YRVllg=,KDcckdOdPX/Uve2c5f7RNbAQrB27wyafOtzqX260DT4=
3,j0URLNNWbjl9ypDnsdULQAhvQ4roRAFjP0kGszyEXXw=,PBpDKzobFwTYitup5cJrqGpOK6iVZJvV6r42rmEbbZY=
4,kOnOAfd4lRrulCE9uv9ekwUcP/W9Hs41K1i5U0OLAfo=,+mx1i1sOZ7tc0lLUnrEQjGghLy+l5BH5XrTmY72A9Zc=


## 6. 기상 결측 보완과 rolling feature

기상 feature는 도축일 이전 누적 노출을 보도록 만듭니다. 당일값이 아니라 각 관측지점별로 하루 shift한 뒤 rolling을 계산합니다.

결측 처리 원칙:

- 기온/풍속: 같은 지점에서 1~3일 짧은 결측은 선형보간
- 강수/습도: 같은 날짜의 가까운 관측소 값을 우선 사용
- 남은 결측: 지점-월 median, 전체-월 median, 전체 median 순서
- `ta_max < ta_min`, 습도 0~100 범위 같은 기본 품질 검사 포함

In [8]:
def haversine_np(lat1, lon1, lat2, lon2):
    r = 6371.0088
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


def build_nearest_station_map(station_df, k=8):
    if station_df is None:
        return {}
    coord = station_df[['stn', '위도', '경도']].dropna().drop_duplicates('stn').copy()
    coord['stn'] = pd.to_numeric(coord['stn'], errors='coerce')
    coord = coord.dropna(subset=['stn'])
    coord['stn'] = coord['stn'].astype(int)
    values = coord[['stn', '위도', '경도']].to_numpy()
    nearest = {}
    for stn, lat, lon in values:
        dist = haversine_np(lat, lon, values[:, 1], values[:, 2])
        order = np.argsort(dist)
        nearest[int(stn)] = [int(values[i, 0]) for i in order if int(values[i, 0]) != int(stn)][:k]
    return nearest


def spatial_fill_same_date(weather_df, value_col, nearest_map, max_neighbors=8):
    if not nearest_map or value_col not in weather_df.columns:
        return weather_df[value_col]
    out = weather_df[value_col].copy()
    missing_idx = out[out.isna()].index
    if len(missing_idx) == 0:
        return out
    pivot = weather_df.pivot_table(index='date', columns='stn', values=value_col, aggfunc='first')
    dates = weather_df.loc[missing_idx, 'date'].to_numpy()
    stns = weather_df.loc[missing_idx, 'stn'].astype('Int64').to_numpy()
    fill_values = []
    for dt, stn in zip(dates, stns):
        value = np.nan
        if pd.notna(stn) and dt in pivot.index:
            for neighbor in nearest_map.get(int(stn), [])[:max_neighbors]:
                if neighbor in pivot.columns:
                    candidate = pivot.at[dt, neighbor]
                    if pd.notna(candidate):
                        value = candidate
                        break
        fill_values.append(value)
    out.loc[missing_idx] = out.loc[missing_idx].fillna(pd.Series(fill_values, index=missing_idx))
    return out


def fill_station_month_then_global(weather_df, value_cols):
    out = weather_df.copy()
    out['month'] = out['date'].dt.month
    for col in value_cols:
        out[col] = out[col].fillna(out.groupby(['stn', 'month'])[col].transform('median'))
        out[col] = out[col].fillna(out.groupby('month')[col].transform('median'))
        out[col] = out[col].fillna(out[col].median())
    return out.drop(columns=['month'])


def prepare_weather_daily(weather_df, station_df=None):
    if weather_df is None:
        return None
    out = normalize_missing(weather_df)
    out['date'] = pd.to_datetime(out['date'], errors='coerce')
    out['stn'] = pd.to_numeric(out['stn'], errors='coerce').astype('Int64')
    weather_cols = ['ta_max', 'rn_day', 'ta_min', 'rhm_avg', 'ws_davg']
    for col in weather_cols:
        out[col] = pd.to_numeric(out[col], errors='coerce')
        out[f'{col}_was_missing'] = out[col].isna().astype('int8')
    out = out.sort_values(['stn', 'date']).reset_index(drop=True)
    for col in ['ta_max', 'ta_min', 'ws_davg']:
        out[col] = out.groupby('stn')[col].transform(lambda s: s.interpolate(method='linear', limit=3, limit_area='inside'))
    if APPLY_SPATIAL_WEATHER_IMPUTE:
        nearest_map = build_nearest_station_map(station_df)
        for col in ['rn_day', 'rhm_avg', 'ta_max', 'ta_min', 'ws_davg']:
            out[col] = spatial_fill_same_date(out, col, nearest_map)
    out = fill_station_month_then_global(out, weather_cols)
    swapped = out['ta_max'] < out['ta_min']
    if swapped.any():
        tmp = out.loc[swapped, 'ta_max'].copy()
        out.loc[swapped, 'ta_max'] = out.loc[swapped, 'ta_min']
        out.loc[swapped, 'ta_min'] = tmp
    out['rhm_avg'] = out['rhm_avg'].clip(0, 100)
    out['rn_day'] = out['rn_day'].clip(lower=0)
    out['ws_davg'] = out['ws_davg'].clip(lower=0)
    return out


def build_weather_rolling_features(weather_daily, windows):
    if weather_daily is None:
        return None
    w = weather_daily.sort_values(['stn', 'date']).copy()
    w['ta_avg'] = (w['ta_max'] + w['ta_min']) / 2
    w['thi'] = (1.8 * w['ta_avg'] + 32) - (0.55 - 0.0055 * w['rhm_avg']) * (1.8 * w['ta_avg'] - 26)
    w['rain_day'] = (w['rn_day'] > 0).astype('int8')
    w['hot_day'] = (w['ta_max'] >= 30).astype('int8')
    w['heatwave_day'] = (w['ta_max'] >= 33).astype('int8')
    w['tropical_night'] = (w['ta_min'] >= 25).astype('int8')
    w['thi72_day'] = (w['thi'] >= 72).astype('int8')
    w['thi78_day'] = (w['thi'] >= 78).astype('int8')
    base_cols = ['ta_max', 'ta_min', 'ta_avg', 'rhm_avg', 'ws_davg', 'thi']
    sum_cols = ['rn_day', 'rain_day', 'hot_day', 'heatwave_day', 'tropical_night', 'thi72_day', 'thi78_day']
    miss_cols = [c for c in w.columns if c.endswith('_was_missing')]
    pieces = []
    for stn, g in w.groupby('stn', sort=False):
        g = g.set_index('date').sort_index()
        shifted = g[base_cols + sum_cols + miss_cols].shift(1)
        feat = pd.DataFrame(index=g.index)
        feat['stn'] = stn
        for window in windows:
            min_periods = max(7, int(window * 0.6))
            roll = shifted.rolling(window=window, min_periods=min_periods)
            for col in base_cols:
                feat[f'{col}_mean_{window}d'] = roll[col].mean()
            for col in sum_cols:
                feat[f'{col}_sum_{window}d'] = roll[col].sum()
            for col in miss_cols:
                feat[f'{col}_ratio_{window}d'] = roll[col].mean()
        pieces.append(feat.reset_index())
    return pd.concat(pieces, ignore_index=True).rename(columns={'date': 'ABATT_DATE'})


weather_features = None
if ADD_WEATHER_FEATURES:
    weather_daily = prepare_weather_daily(weather, station)
    print('weather_daily:', weather_daily.shape)
    weather_features = build_weather_rolling_features(weather_daily, WEATHER_WINDOWS)
    weather_features['stn'] = pd.to_numeric(weather_features['stn'], errors='coerce').astype('Int64')
    print('weather_features:', weather_features.shape)
    display(weather_features.head())

weather_daily: (973248, 12)
weather_features: (973248, 56)


,ABATT_DATE,stn,ta_max_mean_30d,ta_min_mean_30d,ta_avg_mean_30d,rhm_avg_mean_30d,ws_davg_mean_30d,thi_mean_30d,rn_day_sum_30d,rain_day_sum_30d,hot_day_sum_30d,heatwave_day_sum_30d,tropical_night_sum_30d,thi72_day_sum_30d,thi78_day_sum_30d,ta_max_was_missing_ratio_30d,rn_day_was_missing_ratio_30d,ta_min_was_missing_ratio_30d,rhm_avg_was_missing_ratio_30d,ws_davg_was_missing_ratio_30d,ta_max_mean_90d,ta_min_mean_90d,ta_avg_mean_90d,rhm_avg_mean_90d,ws_davg_mean_90d,thi_mean_90d,rn_day_sum_90d,rain_day_sum_90d,hot_day_sum_90d,heatwave_day_sum_90d,tropical_night_sum_90d,thi72_day_sum_90d,thi78_day_sum_90d,ta_max_was_missing_ratio_90d,rn_day_was_missing_ratio_90d,ta_min_was_missing_ratio_90d,rhm_avg_was_missing_ratio_90d,ws_davg_was_missing_ratio_90d,ta_max_mean_180d,ta_min_mean_180d,ta_avg_mean_180d,rhm_avg_mean_180d,ws_davg_mean_180d,thi_mean_180d,rn_day_sum_180d,rain_day_sum_180d,hot_day_sum_180d,heatwave_day_sum_180d,tropical_night_sum_180d,thi72_day_sum_180d,thi78_day_sum_180d,ta_max_was_missing_ratio_180d,rn_day_was_missing_ratio_180d,ta_min_was_missing_ratio_180d,rhm_avg_was_missing_ratio_180d,ws_davg_was_missing_ratio_180d
0,2020-01-01,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-02,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-01-03,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-01-04,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-01-05,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. 기상 feature 결합

기상 rolling feature는 `stn + ABATT_DATE` 기준으로 결합합니다. rolling은 이미 하루 shift를 했기 때문에 도축일 당일값이 아니라 도축 전 누적 기상입니다.

In [9]:
def merge_weather_features(df, weather_features_df):
    if weather_features_df is None:
        return df.copy()
    out = df.copy()
    out['stn'] = pd.to_numeric(out['stn'], errors='coerce').astype('Int64')
    return out.merge(weather_features_df, on=['stn', 'ABATT_DATE'], how='left')


for key in parts:
    parts[key] = merge_weather_features(parts[key], weather_features)

for key, df in parts.items():
    print(key, df.shape)

train (1686089, 85)
valid (361305, 85)
test (361305, 85)


## 8. Train 기준 결측 보완

이 셀은 가장 중요합니다. validation/test 정보가 전처리 기준값에 섞이지 않도록, 모든 중앙값/최빈값/frequency mapping은 train split에서만 계산합니다.

처리 순서:

1. 도체 수치형 결측 개수를 `carcass_missing_count`로 저장
2. `WGRADE`는 성별 + 나이 + 도축연도 기준 최빈값으로 보완
3. 도체 수치형은 성별 + 나이 + 도축연도 기준 중앙값으로 보완
4. `GROWTH` 보완 후 `growth_ge8` 생성
5. `COST_AMT`는 성별 + 도축연월 + `WGRADE` 기준 중앙값으로 보완
6. `cow_year`, `AREA`는 지역 기준 중앙값으로 보완하고 log feature 생성
7. KPN 빈도는 train split에서만 계산

In [10]:
def _key_frame(df, group_cols):
    keys = df[list(group_cols)].copy()
    for col in group_cols:
        keys[col] = keys[col].astype('string').fillna('__NA__')
    return keys


def fit_group_stat(df, value_col, group_cols, kind='median', numeric=True):
    keys = _key_frame(df, group_cols)
    values = pd.to_numeric(df[value_col], errors='coerce') if numeric else df[value_col].astype('string').replace('__NA__', np.nan)
    work = keys.copy()
    work['_value'] = values
    work = work.dropna(subset=['_value'])
    if work.empty:
        return pd.DataFrame(columns=list(group_cols) + ['_fill'])
    if kind == 'median':
        return work.groupby(list(group_cols), as_index=False)['_value'].median().rename(columns={'_value': '_fill'})
    if kind == 'mode':
        return work.groupby(list(group_cols), as_index=False)['_value'].agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan).rename(columns={'_value': '_fill'})
    raise ValueError(kind)


def fit_fallback_imputer(train_df, value_col, group_chain, kind='median', numeric=True):
    if numeric:
        observed = pd.to_numeric(train_df[value_col], errors='coerce').dropna()
        global_fill = observed.median() if len(observed) else 0.0
    else:
        observed = train_df[value_col].dropna().astype('string')
        global_fill = observed.mode().iloc[0] if not observed.mode().empty else 'UNKNOWN'
    return {
        'value_col': value_col,
        'numeric': numeric,
        'global_fill': global_fill,
        'tables': [(cols, fit_group_stat(train_df, value_col, cols, kind=kind, numeric=numeric)) for cols in group_chain],
    }


def apply_fallback_imputer(df, spec):
    out = df.copy()
    col = spec['value_col']
    out[col] = pd.to_numeric(out[col], errors='coerce') if spec['numeric'] else out[col].astype('string').replace('__NA__', np.nan)
    for group_cols, table in spec['tables']:
        missing = out[col].isna()
        if not missing.any() or table.empty:
            continue
        keys = _key_frame(out.loc[missing], group_cols)
        fill = keys.merge(table, on=list(group_cols), how='left')['_fill'].to_numpy()
        out.loc[missing, col] = out.loc[missing, col].fillna(pd.Series(fill, index=out.index[missing]))
    out[col] = out[col].fillna(spec['global_fill'])
    return out


def fit_and_apply_imputers(parts):
    fitted = {}
    out_parts = {k: v.copy() for k, v in parts.items()}

    for key in out_parts:
        out_parts[key]['carcass_missing_count'] = out_parts[key][CARCASS_NUMERIC_COLS].isna().sum(axis=1)

    wgrade_spec = fit_fallback_imputer(out_parts['train'], 'WGRADE', [['JUDGE_SEX', 'AGE', 'abatt_year'], ['JUDGE_SEX', 'AGE'], ['JUDGE_SEX']], kind='mode', numeric=False)
    fitted['WGRADE'] = wgrade_spec
    for key in out_parts:
        out_parts[key] = apply_fallback_imputer(out_parts[key], wgrade_spec)

    for col in CARCASS_NUMERIC_COLS:
        spec = fit_fallback_imputer(out_parts['train'], col, [['JUDGE_SEX', 'AGE', 'abatt_year'], ['JUDGE_SEX', 'AGE'], ['JUDGE_SEX']], kind='median', numeric=True)
        fitted[col] = spec
        for key in out_parts:
            out_parts[key] = apply_fallback_imputer(out_parts[key], spec)

    for key in out_parts:
        out_parts[key]['growth_ge8'] = (pd.to_numeric(out_parts[key]['GROWTH'], errors='coerce') >= 8).astype('int8')

    cost_spec = fit_fallback_imputer(out_parts['train'], 'COST_AMT', [['JUDGE_SEX', 'abatt_ym', 'WGRADE'], ['JUDGE_SEX', 'abatt_ym'], ['JUDGE_SEX']], kind='median', numeric=True)
    fitted['COST_AMT'] = cost_spec
    for key in out_parts:
        out_parts[key] = apply_fallback_imputer(out_parts[key], cost_spec)

    for col in ['cow_year', 'AREA']:
        if col in out_parts['train'].columns:
            spec = fit_fallback_imputer(out_parts['train'], col, [['sigungu'], ['sido']], kind='median', numeric=True)
            fitted[col] = spec
            for key in out_parts:
                out_parts[key] = apply_fallback_imputer(out_parts[key], spec)

    for key in out_parts:
        if 'cow_year' not in out_parts[key].columns:
            out_parts[key]['cow_year'] = np.nan
        if 'AREA' not in out_parts[key].columns:
            out_parts[key]['AREA'] = np.nan
        out_parts[key]['log_cow_year'] = np.log1p(pd.to_numeric(out_parts[key]['cow_year'], errors='coerce').clip(lower=0))
        out_parts[key]['log_area'] = np.log1p(pd.to_numeric(out_parts[key]['AREA'], errors='coerce').clip(lower=0))

    train_kpn = out_parts['train']['KPN_NO'].dropna().astype('string') if 'KPN_NO' in out_parts['train'].columns else pd.Series(dtype='string')
    kpn_freq_map = train_kpn.value_counts().to_dict()
    fitted['kpn_freq_map'] = kpn_freq_map
    for key in out_parts:
        if 'KPN_NO' not in out_parts[key].columns:
            out_parts[key]['KPN_NO'] = np.nan
        kpn = out_parts[key]['KPN_NO'].astype('string')
        out_parts[key]['kpn_known'] = kpn.notna().astype('int8')
        out_parts[key]['kpn_freq'] = kpn.map(kpn_freq_map).fillna(0).astype('int32')
        out_parts[key]['log_kpn_freq'] = np.log1p(out_parts[key]['kpn_freq'])

    return out_parts, fitted


parts, fitted_preprocess = fit_and_apply_imputers(parts)

for key, df in parts.items():
    print(key, df.shape, 'remaining missing core:', int(df[CARCASS_NUMERIC_COLS + ['COST_AMT', 'WGRADE']].isna().sum().sum()))
    display(df[['carcass_missing_count', 'GROWTH', 'growth_ge8', 'COST_AMT', 'cow_year', 'log_area', 'kpn_known', 'kpn_freq']].head())

train (1686089, 92) remaining missing core: 0


,carcass_missing_count,GROWTH,growth_ge8,COST_AMT,cow_year,log_area,kpn_known,kpn_freq
0,0,7.0,0,11037.0,89.0,6.993015,1,1051
1,0,5.0,0,17323.0,9.0,4.633952,1,10276
2,0,9.0,1,11000.0,563.0,6.693324,0,0
3,0,3.0,0,16182.0,69.0,7.507141,1,4135
4,0,9.0,1,9335.0,41.0,6.897422,1,640


valid (361305, 92) remaining missing core: 0


,carcass_missing_count,GROWTH,growth_ge8,COST_AMT,cow_year,log_area,kpn_known,kpn_freq
0,0,7.0,0,10400.0,10.0,5.444580,1,2096
1,0,6.0,0,20410.0,86.0,6.613921,1,2611
2,0,7.0,0,16590.0,44.0,7.031397,1,6380
3,0,9.0,1,19000.0,178.0,8.207066,1,2890
4,0,5.0,0,16370.0,59.0,7.467942,0,0


test (361305, 92) remaining missing core: 0


,carcass_missing_count,GROWTH,growth_ge8,COST_AMT,cow_year,log_area,kpn_known,kpn_freq
0,0,7.0,0,12000.0,34.0,7.535297,1,2397
1,0,7.0,0,15560.0,103.0,7.711665,0,0
2,0,4.0,0,11208.0,104.0,6.695898,0,0
3,0,3.0,0,18899.0,124.0,7.304771,1,1934
4,0,2.0,0,13899.0,108.0,7.090910,0,0


## 9. 모델 feature 구성

최종 점수용 모델은 도체 판정 변수와 가격까지 포함합니다. 이유는 대회 정량 평가지표가 Macro-F1이고, 이미 확인한 결과 `COST_AMT`와 도체 판정 변수들이 매우 강한 예측력을 갖기 때문입니다.

다만 보고서용 해석에서는 `COST_AMT` 제외 모델이나 일부 판정 변수 축소 모델을 별도로 둘 수 있습니다. 이 노트북의 V1.1은 먼저 `통합모델에 성별 포함 X` 조건의 성능 기준선을 만드는 데 집중합니다.

In [11]:
def get_feature_columns(df):
    numeric_features = [
        'WEIGHT', 'BACKFAT', 'REA', 'WINDEX', 'INSFAT', 'YUKSAK', 'FATSAK', 'TISSUE', 'GROWTH',
        'COST_AMT', 'AGE', 'judge_delay_days', 'carcass_missing_count', 'growth_ge8',
        'cow_year', 'log_cow_year', 'log_area', 'kpn_known', 'kpn_freq', 'log_kpn_freq'
    ]
    weather_numeric = [c for c in df.columns if any(c.endswith(f'_{w}d') for w in WEATHER_WINDOWS)]
    numeric_features += weather_numeric
    numeric_features = [c for c in numeric_features if c in df.columns]

    # V1.1: 통합모델에 성별 포함 X. JUDGE_SEX는 split/평가용으로만 쓰고 feature에서는 제외한다.
    categorical_features = ['WGRADE', 'sido', 'sigungu', 'stn', 'abatt_month', 'abatt_season']
    categorical_features = [c for c in categorical_features if c in df.columns]
    return numeric_features, categorical_features


numeric_features, categorical_features = get_feature_columns(parts['train'])
features = numeric_features + categorical_features

print('numeric_features:', len(numeric_features))
print(numeric_features[:100])
print('categorical_features:', categorical_features)

with open(OUTPUT_DIR / 'model_v3_feature_columns.json', 'w', encoding='utf-8') as f:
    json.dump({'numeric_features': numeric_features, 'categorical_features': categorical_features}, f, ensure_ascii=False, indent=2)

numeric_features: 74
['WEIGHT', 'BACKFAT', 'REA', 'WINDEX', 'INSFAT', 'YUKSAK', 'FATSAK', 'TISSUE', 'GROWTH', 'COST_AMT', 'AGE', 'judge_delay_days', 'carcass_missing_count', 'growth_ge8', 'cow_year', 'log_cow_year', 'log_area', 'kpn_known', 'kpn_freq', 'log_kpn_freq', 'ta_max_mean_30d', 'ta_min_mean_30d', 'ta_avg_mean_30d', 'rhm_avg_mean_30d', 'ws_davg_mean_30d', 'thi_mean_30d', 'rn_day_sum_30d', 'rain_day_sum_30d', 'hot_day_sum_30d', 'heatwave_day_sum_30d', 'tropical_night_sum_30d', 'thi72_day_sum_30d', 'thi78_day_sum_30d', 'ta_max_was_missing_ratio_30d', 'rn_day_was_missing_ratio_30d', 'ta_min_was_missing_ratio_30d', 'rhm_avg_was_missing_ratio_30d', 'ws_davg_was_missing_ratio_30d', 'ta_max_mean_90d', 'ta_min_mean_90d', 'ta_avg_mean_90d', 'rhm_avg_mean_90d', 'ws_davg_mean_90d', 'thi_mean_90d', 'rn_day_sum_90d', 'rain_day_sum_90d', 'hot_day_sum_90d', 'heatwave_day_sum_90d', 'tropical_night_sum_90d', 'thi72_day_sum_90d', 'thi78_day_sum_90d', 'ta_max_was_missing_ratio_90d', 'rn_day_was_m

## 10. 모델 정의

모델별 처리 방식은 약간 다릅니다.

- `RandomForest`, `LinearSVM`, `XGBoost`: 범주형을 one-hot encoding합니다.
- `CatBoost`: 범주형 변수를 native categorical feature로 직접 전달합니다. CatBoost의 장점이 범주형 처리라서 이 방식이 더 자연스럽습니다.
- `XGBoost`: 문자열 label을 바로 받지 못하는 경우가 많아서 내부에서 `LabelEncoder`로 target을 정수화한 뒤 다시 원래 등급명으로 복원합니다.


In [12]:
class MissingDependencyError(RuntimeError):
    pass


def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=True)


def make_linear_svc(random_state):
    try:
        return LinearSVC(
            C=1.0,
            class_weight='balanced',
            max_iter=3000,
            dual='auto',
            random_state=random_state,
        )
    except TypeError:
        return LinearSVC(
            C=1.0,
            class_weight='balanced',
            max_iter=3000,
            random_state=random_state,
        )


def build_sparse_preprocess(scale_numeric=False):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler(with_mean=False)))
    numeric_transformer = Pipeline(steps=numeric_steps)
    categorical_transformer = Pipeline(steps=[
        ('to_string', FunctionTransformer(lambda x: x.astype('string').fillna('UNKNOWN').astype(str), feature_names_out='one-to-one')),
        ('onehot', make_onehot_encoder()),
    ])
    return ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features),
        ],
        remainder='drop',
    )


class LabelEncodedClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator):
        self.estimator = estimator

    def fit(self, X, y):
        self.label_encoder_ = LabelEncoder()
        y_encoded = self.label_encoder_.fit_transform(y)
        self.estimator_ = clone(self.estimator)
        self.estimator_.fit(X, y_encoded)
        self.classes_ = self.label_encoder_.classes_
        return self

    def predict(self, X):
        pred = self.estimator_.predict(X)
        pred = np.asarray(pred).astype(int)
        return self.label_encoder_.inverse_transform(pred)


class CatBoostNativeModel:
    def __init__(self, random_state=RANDOM_STATE):
        self.random_state = random_state

    def _prepare_fit_frame(self, X):
        out = X[features].copy()
        self.numeric_medians_ = {}
        for col in numeric_features:
            out[col] = pd.to_numeric(out[col], errors='coerce')
            med = out[col].median()
            if pd.isna(med):
                med = 0.0
            self.numeric_medians_[col] = med
            out[col] = out[col].fillna(med).astype('float32')
        for col in categorical_features:
            out[col] = out[col].astype('string').fillna('UNKNOWN').astype(str)
        self.cat_feature_indices_ = [out.columns.get_loc(c) for c in categorical_features]
        return out

    def _prepare_predict_frame(self, X):
        out = X[features].copy()
        for col in numeric_features:
            out[col] = pd.to_numeric(out[col], errors='coerce')
            out[col] = out[col].fillna(self.numeric_medians_.get(col, 0.0)).astype('float32')
        for col in categorical_features:
            out[col] = out[col].astype('string').fillna('UNKNOWN').astype(str)
        return out

    def fit(self, X, y):
        if CatBoostClassifier is None:
            raise MissingDependencyError('catboost is not installed or import failed.')
        x_fit = self._prepare_fit_frame(X)
        self.model_ = CatBoostClassifier(
            iterations=CATBOOST_ITERATIONS,
            depth=6,
            learning_rate=0.06,
            loss_function='MultiClass',
            auto_class_weights='Balanced',
            random_seed=self.random_state,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        )
        self.model_.fit(x_fit, y, cat_features=self.cat_feature_indices_)
        return self

    def predict(self, X):
        x_pred = self._prepare_predict_frame(X)
        pred = self.model_.predict(x_pred)
        return np.asarray(pred).reshape(-1)


def build_model(model_name, random_state=RANDOM_STATE):
    if model_name == 'random_forest':
        model = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            random_state=random_state,
            n_jobs=-1,
            class_weight='balanced_subsample',
        )
        return Pipeline(steps=[('preprocess', build_sparse_preprocess()), ('model', model)])

    if model_name == 'linear_svm':
        return Pipeline(steps=[
            ('preprocess', build_sparse_preprocess(scale_numeric=True)),
            ('model', make_linear_svc(random_state)),
        ])

    if model_name == 'xgboost':
        if XGBClassifier is None:
            raise MissingDependencyError('xgboost is not installed or import failed.')
        xgb = XGBClassifier(
            n_estimators=XGB_N_ESTIMATORS,
            max_depth=6,
            learning_rate=0.06,
            subsample=0.85,
            colsample_bytree=0.85,
            objective='multi:softmax',
            eval_metric='mlogloss',
            tree_method='hist',
            n_jobs=-1,
            random_state=random_state,
            verbosity=1,
        )
        return Pipeline(steps=[
            ('preprocess', build_sparse_preprocess()),
            ('model', LabelEncodedClassifier(xgb)),
        ])

    if model_name == 'catboost':
        return CatBoostNativeModel(random_state=random_state)

    raise ValueError(f'Unknown model_name: {model_name}')


print('MODEL_CANDIDATES:', MODEL_CANDIDATES)
print('features:', len(features), 'numeric:', len(numeric_features), 'categorical:', len(categorical_features))


MODEL_CANDIDATES: ['random_forest', 'linear_svm', 'xgboost', 'catboost']
features: 80 numeric: 74 categorical: 6


## 11. 모델 비교 실행

각 모델을 `unified`와 `gender_split` 두 전략으로 비교합니다.

SVM은 기본 설정에서 train 최대 50만 행만 사용합니다. 결과표의 `train_rows_used`를 확인하세요.


In [13]:
X_train = parts['train'][features].copy()
y_train = parts['train'][TARGET].copy()
X_valid = parts['valid'][features].copy()
y_valid = parts['valid'][TARGET].copy()
X_test = parts['test'][features].copy()
y_test = parts['test'][TARGET].copy()

comparison_rows = []
valid_predictions = parts['valid'][['CATTLE_NO', 'FARM_UNIQUE_NO', 'JUDGE_SEX', 'ABATT_DATE', TARGET]].copy()
test_predictions = parts['test'][['CATTLE_NO', 'FARM_UNIQUE_NO', 'JUDGE_SEX', 'ABATT_DATE', TARGET]].copy()
trained_unified_models = {}


def maybe_sample_for_model(model_name, x, y, random_state):
    if model_name != 'linear_svm':
        return x, y, len(y), False
    if LINEAR_SVM_TRAIN_SAMPLE_N is None or len(y) <= LINEAR_SVM_TRAIN_SAMPLE_N:
        return x, y, len(y), False
    sample_idx, _ = train_test_split(
        x.index,
        train_size=LINEAR_SVM_TRAIN_SAMPLE_N,
        random_state=random_state,
        stratify=y,
    )
    return x.loc[sample_idx], y.loc[sample_idx], len(sample_idx), True


def fit_predict_score(model_name, strategy, random_state, x_tr, y_tr, x_va, y_va, x_te, y_te):
    model = build_model(model_name, random_state=random_state)
    x_fit, y_fit, train_rows_used, sampled = maybe_sample_for_model(model_name, x_tr, y_tr, random_state)
    start = time.time()
    model.fit(x_fit, y_fit)
    valid_pred = model.predict(x_va)
    test_pred = model.predict(x_te)
    elapsed = time.time() - start
    valid_f1 = f1_score(y_va, valid_pred, average='macro')
    test_f1 = f1_score(y_te, test_pred, average='macro')
    comparison_rows.append({
        'model': model_name,
        'strategy': strategy,
        'valid_macro_f1': valid_f1,
        'test_macro_f1': test_f1,
        'fit_predict_seconds': elapsed,
        'train_rows_used': train_rows_used,
        'sampled_train': sampled,
        'status': 'ok',
        'error': '',
    })
    print(f'{model_name:14s} {strategy:12s} valid={valid_f1:.6f} test={test_f1:.6f} rows={train_rows_used:,} time={elapsed:.1f}s')
    return model, valid_pred, test_pred


for model_i, model_name in enumerate(MODEL_CANDIDATES):
    print('\n' + '=' * 90)
    print(f'[{model_name}]')
    valid_pred_unified = None
    test_pred_unified = None

    try:
        if RUN_UNIFIED or RUN_GENDER_SPLIT:
            unified_model, valid_pred_unified, test_pred_unified = fit_predict_score(
                model_name, 'unified', RANDOM_STATE + model_i * 100,
                X_train, y_train, X_valid, y_valid, X_test, y_test,
            )
            trained_unified_models[model_name] = unified_model
            valid_predictions[f'pred_{model_name}_unified'] = valid_pred_unified
            test_predictions[f'pred_{model_name}_unified'] = test_pred_unified

        if RUN_GENDER_SPLIT:
            valid_pred_gender = pd.Series(valid_pred_unified.copy(), index=X_valid.index, name='pred')
            test_pred_gender = pd.Series(test_pred_unified.copy(), index=X_test.index, name='pred')
            gender_start = time.time()
            gender_rows_used = 0
            gender_sampled_any = False

            for sex_i, sex in enumerate(SPLIT_SEXES):
                train_mask = parts['train']['JUDGE_SEX'].eq(sex)
                valid_mask = parts['valid']['JUDGE_SEX'].eq(sex)
                test_mask = parts['test']['JUDGE_SEX'].eq(sex)
                print(f'  - {sex}: train={int(train_mask.sum()):,}, valid={int(valid_mask.sum()):,}, test={int(test_mask.sum()):,}')

                sex_model = build_model(model_name, random_state=RANDOM_STATE + model_i * 100 + sex_i + 1)
                x_fit, y_fit, rows_used, sampled = maybe_sample_for_model(
                    model_name,
                    parts['train'].loc[train_mask, features],
                    parts['train'].loc[train_mask, TARGET],
                    RANDOM_STATE + model_i * 100 + sex_i + 1,
                )
                sex_model.fit(x_fit, y_fit)
                gender_rows_used += rows_used
                gender_sampled_any = gender_sampled_any or sampled

                if valid_mask.any():
                    valid_pred_gender.loc[valid_mask] = sex_model.predict(parts['valid'].loc[valid_mask, features])
                if test_mask.any():
                    test_pred_gender.loc[test_mask] = sex_model.predict(parts['test'].loc[test_mask, features])

            elapsed = time.time() - gender_start
            valid_f1 = f1_score(y_valid, valid_pred_gender, average='macro')
            test_f1 = f1_score(y_test, test_pred_gender, average='macro')
            comparison_rows.append({
                'model': model_name,
                'strategy': 'gender_split',
                'valid_macro_f1': valid_f1,
                'test_macro_f1': test_f1,
                'fit_predict_seconds': elapsed,
                'train_rows_used': gender_rows_used,
                'sampled_train': gender_sampled_any,
                'status': 'ok',
                'error': '',
            })
            valid_predictions[f'pred_{model_name}_gender_split'] = valid_pred_gender.to_numpy()
            test_predictions[f'pred_{model_name}_gender_split'] = test_pred_gender.to_numpy()
            print(f'{model_name:14s} gender_split valid={valid_f1:.6f} test={test_f1:.6f} rows={gender_rows_used:,} time={elapsed:.1f}s')

    except Exception as e:
        comparison_rows.append({
            'model': model_name,
            'strategy': 'failed',
            'valid_macro_f1': np.nan,
            'test_macro_f1': np.nan,
            'fit_predict_seconds': np.nan,
            'train_rows_used': 0,
            'sampled_train': False,
            'status': 'failed',
            'error': repr(e),
        })
        print(f'FAILED {model_name}: {repr(e)}')

comparison = pd.DataFrame(comparison_rows).sort_values(['test_macro_f1', 'valid_macro_f1'], ascending=False)
comparison.to_csv(OUTPUT_DIR / 'model_v3_metrics.csv', index=False, encoding='utf-8-sig')
valid_predictions.to_csv(OUTPUT_DIR / 'model_v3_validation_predictions.csv', index=False, encoding='utf-8-sig')
test_predictions.to_csv(OUTPUT_DIR / 'model_v3_test_predictions.csv', index=False, encoding='utf-8-sig')
display(comparison)



[random_forest]
random_forest  unified      valid=0.949371 test=0.949530 rows=1,686,089 time=2051.4s
  - 암: train=822,103, valid=176,078, test=176,233
  - 거세: train=855,745, valid=183,400, test=183,320
random_forest  gender_split valid=0.948613 test=0.948850 rows=1,677,848 time=1263.7s

[linear_svm]
linear_svm     unified      valid=0.785745 test=0.786252 rows=500,000 time=650.5s
  - 암: train=822,103, valid=176,078, test=176,233
  - 거세: train=855,745, valid=183,400, test=183,320
linear_svm     gender_split valid=0.809819 test=0.809539 rows=1,000,000 time=1675.1s

[xgboost]
xgboost        unified      valid=0.985170 test=0.984597 rows=1,686,089 time=505.9s
  - 암: train=822,103, valid=176,078, test=176,233
  - 거세: train=855,745, valid=183,400, test=183,320
xgboost        gender_split valid=0.985401 test=0.984853 rows=1,677,848 time=505.3s

[catboost]
FAILED catboost: CatBoostError('bad allocation')


,model,strategy,valid_macro_f1,test_macro_f1,fit_predict_seconds,train_rows_used,sampled_train,status,error
5,xgboost,gender_split,0.985401,0.984853,505.316168,1677848,False,ok,
4,xgboost,unified,0.985170,0.984597,505.935413,1686089,False,ok,
0,random_forest,unified,0.949371,0.949530,2051.402250,1686089,False,ok,
1,random_forest,gender_split,0.948613,0.948850,1263.669311,1677848,False,ok,
3,linear_svm,gender_split,0.809819,0.809539,1675.099775,1000000,True,ok,
2,linear_svm,unified,0.785745,0.786252,650.462291,500000,True,ok,
6,catboost,failed,NaN,NaN,NaN,0,False,failed,CatBoostError('bad allocation')


## 12. 클래스별/성별별 리포트 저장

Macro-F1뿐 아니라 `3A`, `3B`, `2A`, `2B`, `등외` 같은 취약 클래스가 어떻게 변하는지 확인합니다.


In [14]:
def macro_f1_by_group(y_true, y_pred, group):
    rows = []
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    group = pd.Series(group).reset_index(drop=True)
    for g, idx in group.groupby(group).groups.items():
        rows.append({
            'group': g,
            'n': len(idx),
            'macro_f1': f1_score(y_true.iloc[list(idx)], y_pred.iloc[list(idx)], average='macro')
        })
    return pd.DataFrame(rows).sort_values('group')


class_rows = []
sex_rows = []

for col in [c for c in valid_predictions.columns if c.startswith('pred_')]:
    tag = col.replace('pred_', '')
    pred = valid_predictions[col]

    report = pd.DataFrame(classification_report(y_valid, pred, output_dict=True, zero_division=0)).T
    report.to_csv(OUTPUT_DIR / f'model_v3_valid_class_report_{tag}.csv', encoding='utf-8-sig')
    macro = report.loc['macro avg'].to_dict()
    macro.update({'model_strategy': tag})
    class_rows.append(macro)

    sex_report = macro_f1_by_group(y_valid, pred, parts['valid']['JUDGE_SEX'])
    sex_report.insert(0, 'model_strategy', tag)
    sex_rows.append(sex_report)

class_summary = pd.DataFrame(class_rows).sort_values('f1-score', ascending=False)
sex_summary = pd.concat(sex_rows, ignore_index=True) if sex_rows else pd.DataFrame()

class_summary.to_csv(OUTPUT_DIR / 'model_v3_valid_macro_summary.csv', index=False, encoding='utf-8-sig')
sex_summary.to_csv(OUTPUT_DIR / 'model_v3_valid_sex_report.csv', index=False, encoding='utf-8-sig')

display(class_summary)
display(sex_summary)


,precision,recall,f1-score,support,model_strategy
5,0.985834,0.984994,0.985401,361305.0,xgboost_gender_split
4,0.985473,0.984900,0.985170,361305.0,xgboost_unified
0,0.952674,0.947332,0.949371,361305.0,random_forest_unified
1,0.952045,0.946372,0.948613,361305.0,random_forest_gender_split
3,0.810840,0.813405,0.809819,361305.0,linear_svm_gender_split
2,0.793516,0.788278,0.785745,361305.0,linear_svm_unified


,model_strategy,group,n,macro_f1
0,random_forest_unified,거세,183400,0.898744
1,random_forest_unified,수,1827,0.911573
2,random_forest_unified,암,176078,0.955337
3,random_forest_gender_split,거세,183400,0.931017
4,random_forest_gender_split,수,1827,0.911573
5,random_forest_gender_split,암,176078,0.945067
6,linear_svm_unified,거세,183400,0.755128
7,linear_svm_unified,수,1827,0.571888
8,linear_svm_unified,암,176078,0.763052
9,linear_svm_gender_split,거세,183400,0.795787


## 13. 최종 후보 오답 분석

validation Macro-F1 기준 최상위 모델의 confusion matrix와 주요 오답을 저장합니다.


In [15]:
ok_comparison = comparison[comparison['status'].eq('ok')].copy()
if ok_comparison.empty:
    print('성공한 모델이 없습니다.')
else:
    best = ok_comparison.sort_values(['valid_macro_f1', 'test_macro_f1'], ascending=False).iloc[0]
    best_col = f"pred_{best['model']}_{best['strategy']}"
    print('Best by validation Macro-F1')
    display(best.to_frame().T)

    labels = sorted(y_valid.unique())
    cm = pd.DataFrame(
        confusion_matrix(y_valid, valid_predictions[best_col], labels=labels),
        index=[f'true_{x}' for x in labels],
        columns=[f'pred_{x}' for x in labels],
    )
    cm.to_csv(OUTPUT_DIR / 'model_v3_best_valid_confusion_matrix.csv', encoding='utf-8-sig')
    display(cm)

    top_errors = (
        valid_predictions.loc[valid_predictions[TARGET].ne(valid_predictions[best_col]), [TARGET, best_col]]
        .groupby([TARGET, best_col])
        .size()
        .sort_values(ascending=False)
        .head(30)
        .reset_index(name='count')
    )
    top_errors.to_csv(OUTPUT_DIR / 'model_v3_best_top_errors.csv', index=False, encoding='utf-8-sig')
    display(top_errors)


Best by validation Macro-F1


,model,strategy,valid_macro_f1,test_macro_f1,fit_predict_seconds,train_rows_used,sampled_train,status,error
5,xgboost,gender_split,0.985401,0.984853,505.316168,1677848,False,ok,


,pred_1++A,pred_1++B,pred_1++C,pred_1+A,pred_1+B,pred_1+C,pred_1A,pred_1B,pred_1C,pred_2A,pred_2B,pred_2C,pred_3A,pred_3B,pred_3C,pred_등외
true_1++A,30972,82,0,0,0,0,0,0,0,0,0,0,0,0,0,0
true_1++B,128,47809,1,0,0,0,0,0,0,0,0,0,0,0,0,0
true_1++C,0,6,19344,0,0,0,0,0,0,0,0,0,0,0,0,0
true_1+A,1,0,0,24966,120,0,0,0,0,0,0,0,0,0,0,0
true_1+B,0,0,0,276,46404,2,0,0,1,0,0,0,0,0,0,0
true_1+C,0,0,0,0,13,19500,0,0,1,0,0,0,0,0,0,0
true_1A,0,0,0,0,1,0,24991,234,0,0,0,0,0,0,0,0
true_1B,0,0,0,1,1,0,462,44426,4,0,0,0,0,0,0,0
true_1C,0,0,0,0,0,0,0,32,17514,0,0,0,0,0,0,0
true_2A,0,0,0,0,0,0,5,0,0,18630,476,0,0,0,0,0


,LAST_GRADE,pred_xgboost_gender_split,count
0,2B,2A,722
1,2A,2B,476
2,1B,1A,462
3,3B,3A,441
4,3A,3B,404
5,1+B,1+A,276
6,1A,1B,234
7,1++B,1++A,128
8,1+A,1+B,120
9,3C,3B,103


## 14. 해석 가이드

- `RandomForest`가 강하면 현재 v1_1 계열의 bagging tree 접근이 안정적인 것입니다.
- `XGBoost`가 더 강하면 순차적으로 오차를 보정하는 boosting 구조가 등급 경계 구분에 더 유리한 것입니다.
- `CatBoost`가 더 강하면 지역, 지점, 월, 육량등급 같은 categorical feature 처리가 성능 개선에 중요했다는 해석이 가능합니다.
- `LinearSVM`이 낮으면 이 문제는 단순 선형 경계보다 변수 간 비선형 상호작용이 중요하다고 볼 수 있습니다.
